# MaizAI — Disease Classifier Training Pipeline (v2)

This notebook trains a maize leaf disease classifier for the **MaizAI** project, a Master of Engineering dissertation submitted to the Department of Computer Engineering, Faculty of Engineering and Technology, University of Buea (2025/2026 academic year).

## What this notebook does

It produces a single artifact: `maize_classifier.tflite`, a quantised TensorFlow Lite model that classifies an image into one of **five** classes:

| Class | Description |
|---|---|
| Healthy | A maize leaf showing no visible disease |
| Common Rust | Caused by *Puccinia sorghi*; orange-brown pustules |
| Gray Leaf Spot | Caused by *Cercospora zeae-maydis*; rectangular grey lesions |
| Northern Leaf Blight | Caused by *Exserohilum turcicum*; elongated cigar-shaped lesions |
| Not Maize | Anything that is **not** a maize leaf (rejected input) |

The trained model is embedded in the MaizAI Android application and runs entirely on-device, with no internet required for inference.

## What changed in v2

Real-device testing showed the four-class model confidently mislabelling non-maize inputs (a stool, a screen, a flatbread) as a disease. v2 addresses this **open-set recognition** problem at the model level:

1. **A fifth `Not_Maize` class** built from CIFAR-100 negative samples, so the model can reject non-maize inputs itself instead of relying on a heuristic pre-inference gate.
2. **An architecture sweep** comparing MobileNetV2, MobileNetV3-Small and EfficientNet-Lite0 (via EfficientNetB0) under identical splits and hyperparameters.
3. **Confidence calibration analysis** — a reliability diagram and the expected calibration error (ECE).
4. **A formal out-of-distribution (OOD) test** on a held-out negative set.
5. **Selection + export** of the single best architecture to TFLite for the mobile app.

## Architecture (per backbone)

Transfer learning from ImageNet, fine-tuned on the [PlantVillage maize subset](https://www.kaggle.com/datasets/smaranjitghose/corn-or-maize-leaf-disease-dataset) plus the CIFAR-100 negative class, in two phases:

1. **Phase 1** — freeze the backbone, train only the new classification head (10 epochs).
2. **Phase 2** — unfreeze the top backbone layers, fine-tune at a very low learning rate (10 epochs).

After training, the best model is converted to TFLite with **post-training integer quantisation**.

## Expected runtime

On the free Colab **T4 GPU**: approximately **3 hours** end-to-end for all three architectures plus the calibration and OOD evaluation.

## How to run

1. **Runtime → Change runtime type → T4 GPU.** Confirm in the setup cell that the GPU is detected.
2. Run cells in order. Do not skip cells.
3. The final cells download the artifact set listed at the bottom.
4. Commit those files to your repository at `ai/models/`.

## 1. Setup — install dependencies, fix seeds, verify GPU

This cell pins library versions and prepares the environment. Pinned versions guarantee that re-running the notebook months from now produces identical results — a non-negotiable requirement for academic reproducibility.

In [ ]:
# Install pinned versions of every library we use
!pip install -q kagglehub==0.3.4

# Standard library
import os
import json
import random
import shutil
import time

# Numerical and visualisation
import numpy as np
import matplotlib.pyplot as plt

# TensorFlow
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, applications

# scikit-learn for evaluation
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# Kaggle dataset download
import kagglehub

# Fix every random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Verify environment
print(f"TensorFlow version: {tf.__version__}")
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"GPU detected: {gpus[0].name}")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("WARNING: No GPU detected. Training will be much slower on CPU.")
    print("Go to Runtime → Change runtime type → T4 GPU and re-run this cell.")

## 2. Configuration

All hyperparameters and structural choices in one place. If you adjust the training, change only this cell and re-run from here.

### Choices and their rationale

- **`image_size = (224, 224)`** — MobileNetV2's standard input resolution.
- **`batch_size = 32`** — fits comfortably in T4 GPU memory at this resolution.
- **`phase1_learning_rate = 1e-3`** — high enough to train the new head quickly; safe because we are not touching the backbone.
- **`phase2_learning_rate = 1e-5`** — 100× lower than Phase 1. Required during fine-tuning to avoid destroying the ImageNet features.
- **`phase2_unfreeze_from_layer = 100`** — unfreeze approximately the top 30% of MobileNetV2 (which has 154 layers).
- **70/15/15 split** — standard academic practice for datasets of this size.

In [ ]:
CONFIG = {
    # Image and batching
    "image_size": (224, 224),
    "batch_size": 32,

    # Classes — these names must match the dataset folder names exactly.
    # "Northern_Leaf_Blight" is the canonical name; the raw dataset ships the
    # blight folder as "Blight", which the download cell renames to match.
    "num_classes": 5,
    "class_names": [
        "Common_Rust",
        "Gray_Leaf_Spot",
        "Healthy",
        "Northern_Leaf_Blight",
        "Not_Maize",
    ],

    # Architectures compared in the sweep (identical splits/hyperparameters;
    # only the backbone changes).
    "architectures": ["MobileNetV2", "MobileNetV3Small", "EfficientNetLite0"],

    # Phase 1: head-only training
    "phase1_epochs": 10,
    "phase1_learning_rate": 1e-3,

    # Phase 2: fine-tuning
    "phase2_epochs": 10,
    "phase2_learning_rate": 1e-5,
    "phase2_unfreeze_from_layer": 100,

    # Dataset splits
    "train_split": 0.70,
    "val_split": 0.15,
    "test_split": 0.15,

    # Model head
    "dense_units": 128,
    "dropout_rate": 0.2,

    # Out-of-distribution + calibration evaluation
    "ood_test_size": 100,
    "calibration_bins": 15,

    # Output locations (relative to the notebook's working directory)
    "output_dir": "./models",
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)

print("Configuration loaded:")
print(json.dumps(CONFIG, indent=2))

## 3. Download the PlantVillage maize subset

The dataset is hosted on Kaggle by smaranjitghose: roughly 4,188 images across the four classes, total size ~169 MB compressed.

`kagglehub` handles the download, extraction, and caching. The first time this cell runs, the dataset is downloaded; subsequent runs use the cached copy.

### Kaggle authentication on first run

If this is the first time you have used `kagglehub` in this Colab session, you will be prompted to authenticate. Either use the browser flow, or upload `kaggle.json` (from https://www.kaggle.com/settings → API → Create New Token) to `/root/.kaggle/kaggle.json`.

In [ ]:
# Download the dataset (cached after first call)
DATASET_PATH = kagglehub.dataset_download(
    "smaranjitghose/corn-or-maize-leaf-disease-dataset"
)
print(f"Dataset cached at: {DATASET_PATH}")

# The raw dataset ships the blight folder as "Blight"; our canonical class name
# is "Northern_Leaf_Blight". Accept either folder name on disk and normalise.
DISEASE_FOLDER_ALIASES = {
    "Common_Rust": ["Common_Rust"],
    "Gray_Leaf_Spot": ["Gray_Leaf_Spot"],
    "Healthy": ["Healthy"],
    "Northern_Leaf_Blight": ["Northern_Leaf_Blight", "Blight"],
}

def _has_any(path, names):
    return any(os.path.isdir(os.path.join(path, n)) for n in names)

# Locate the directory that holds the four raw disease folders.
DATA_DIR = None
for root, dirs, _ in os.walk(DATASET_PATH):
    if all(_has_any(root, aliases) for aliases in DISEASE_FOLDER_ALIASES.values()):
        DATA_DIR = root
        break

if DATA_DIR is None:
    raise RuntimeError(
        f"Could not find the four disease folders under {DATASET_PATH}. "
        f"Expected one of each: {DISEASE_FOLDER_ALIASES}"
    )
print(f"Class folders located at: {DATA_DIR}")

# Normalise the blight folder name to the canonical Northern_Leaf_Blight so it
# matches CONFIG['class_names'] (image_dataset_from_directory matches by folder
# name). Idempotent: a no-op once already renamed.
for canonical, aliases in DISEASE_FOLDER_ALIASES.items():
    if os.path.isdir(os.path.join(DATA_DIR, canonical)):
        continue
    for alias in aliases:
        src = os.path.join(DATA_DIR, alias)
        if os.path.isdir(src):
            os.rename(src, os.path.join(DATA_DIR, canonical))
            print(f"Renamed dataset folder '{alias}' -> '{canonical}'")
            break

# Verify the four disease classes (the Not_Maize class is added in the next cell).
print("\nDataset structure (disease classes):")
total_images = 0
for cls in ["Common_Rust", "Gray_Leaf_Spot", "Healthy", "Northern_Leaf_Blight"]:
    cls_path = os.path.join(DATA_DIR, cls)
    if not os.path.isdir(cls_path):
        raise RuntimeError(f"Missing class folder: {cls_path}")
    n = len([f for f in os.listdir(cls_path)
             if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    total_images += n
    print(f"  {cls}: {n} images")
print(f"\nTotal disease images: {total_images}")

## 3b. Add the `Not_Maize` negative class (open-set recognition)

We add a fifth class, `Not_Maize`, to teach the model to reject non-maize inputs rather than always producing a confident classification across the four disease classes. The negative class is built from CIFAR-100, filtered to exclude any plant or leaf categories. This is the cleanest way to address the open-set recognition problem revealed during real-device testing.

We also reserve a small held-out set of non-plant images (not used in training) for a formal out-of-distribution test later in the notebook.

In [ ]:
import tensorflow as tf
import numpy as np
from PIL import Image
import os
import shutil

# CIFAR-100 fine labels to exclude (anything plant-like)
CIFAR100_PLANT_LABELS = {
    "apple", "aquarium_fish", "fish", "flatfish", "ray", "shark", "trout",
    "orchid", "poppy", "rose", "sunflower", "tulip",
    "bottle", "bowl", "can", "cup", "plate",
    "apple", "mushroom", "orange", "pear", "sweet_pepper",
    "forest", "mountain", "plain", "sea",
    "maple_tree", "oak_tree", "palm_tree", "pine_tree", "willow_tree",
    "leaf", "leaves",
}

# Download CIFAR-100 (fits in memory; small)
(x_train_cf, y_train_cf), (x_test_cf, y_test_cf) = (
    tf.keras.datasets.cifar100.load_data(label_mode="fine")
)

# Get fine label names
cifar100_label_names = [
    "apple", "aquarium_fish", "baby", "bear", "beaver", "bed", "bee",
    "beetle", "bicycle", "bottle", "bowl", "boy", "bridge", "bus",
    "butterfly", "camel", "can", "castle", "caterpillar", "cattle",
    "chair", "chimpanzee", "clock", "cloud", "cockroach", "couch",
    "crab", "crocodile", "cup", "dinosaur", "dolphin", "elephant",
    "flatfish", "forest", "fox", "girl", "hamster", "house", "kangaroo",
    "keyboard", "lamp", "lawn_mower", "leopard", "lion", "lizard",
    "lobster", "man", "maple_tree", "motorcycle", "mountain", "mouse",
    "mushroom", "oak_tree", "orange", "orchid", "otter", "palm_tree",
    "pear", "pickup_truck", "pine_tree", "plain", "plate", "poppy",
    "porcupine", "possum", "rabbit", "raccoon", "ray", "road", "rocket",
    "rose", "sea", "seal", "shark", "shrew", "skunk", "skyscraper",
    "snail", "snake", "spider", "squirrel", "streetcar", "sunflower",
    "sweet_pepper", "table", "tank", "telephone", "television", "tiger",
    "tractor", "train", "trout", "tulip", "turtle", "wardrobe", "whale",
    "willow_tree", "wolf", "woman", "worm",
]

# Build mask of non-plant labels
non_plant_indices = [
    i for i, name in enumerate(cifar100_label_names)
    if name not in CIFAR100_PLANT_LABELS
]

# Filter training set to non-plant categories
mask = np.isin(y_train_cf.flatten(), non_plant_indices)
x_train_filtered = x_train_cf[mask]

print(f"Filtered CIFAR-100 non-plant images available: {len(x_train_filtered)}")

# Sample 1300 images for the Not_Maize class (matches Common Rust size)
TARGET_NOT_MAIZE = 1300
indices = np.random.RandomState(SEED).choice(
    len(x_train_filtered), TARGET_NOT_MAIZE, replace=False
)
not_maize_samples = x_train_filtered[indices]

# CIFAR-100 images are 32x32; upscale to 224x224 with bilinear interpolation
# and save into the dataset directory so they fit the existing
# image_dataset_from_directory pipeline
NOT_MAIZE_DIR = os.path.join(DATA_DIR, "Not_Maize")
os.makedirs(NOT_MAIZE_DIR, exist_ok=True)

for i, img_array in enumerate(not_maize_samples):
    pil_img = Image.fromarray(img_array)
    pil_img_resized = pil_img.resize((224, 224), Image.BILINEAR)
    pil_img_resized.save(os.path.join(NOT_MAIZE_DIR, f"not_maize_{i:04d}.jpg"))

print(f"Saved {TARGET_NOT_MAIZE} Not_Maize images to {NOT_MAIZE_DIR}")

# Reserve 100 additional non-plant images for the OOD test set
# (not used in training, separate evaluation in the OOD section)
ood_indices = np.random.RandomState(SEED + 1).choice(
    len(x_train_filtered), CONFIG["ood_test_size"], replace=False
)
# Ensure no overlap with the training sample
ood_indices = [idx for idx in ood_indices if idx not in indices][:CONFIG["ood_test_size"]]
OOD_DIR = "./ood_test"
os.makedirs(OOD_DIR, exist_ok=True)
for i, idx in enumerate(ood_indices):
    pil_img = Image.fromarray(x_train_filtered[idx])
    pil_img_resized = pil_img.resize((224, 224), Image.BILINEAR)
    pil_img_resized.save(os.path.join(OOD_DIR, f"ood_{i:03d}.jpg"))

print(f"OOD test set saved: {len(ood_indices)} images at {OOD_DIR}")

# Verify the updated dataset structure
print("\nUpdated dataset structure:")
total = 0
for cls in CONFIG["class_names"]:
    cls_path = os.path.join(DATA_DIR, cls)
    if os.path.isdir(cls_path):
        n = len([f for f in os.listdir(cls_path)
                 if f.lower().endswith((".jpg", ".jpeg", ".png"))])
        total += n
        print(f"  {cls}: {n} images")
print(f"\nTotal images across five classes: {total}")

## 4. Load and split the dataset

We use `image_dataset_from_directory` for efficient on-the-fly loading directly from the folder structure.

### Why one-hot labels

We set `label_mode="categorical"` so labels are one-hot encoded. This pairs naturally with the `categorical_crossentropy` loss.

### How the split works

The full dataset is shuffled once (seeded) and then sliced into three contiguous parts: 70% training, 15% validation, 15% test.

In [ ]:
from tensorflow.keras.preprocessing import image_dataset_from_directory

full_dataset = image_dataset_from_directory(
    DATA_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=CONFIG["class_names"],
    image_size=CONFIG["image_size"],
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    seed=SEED,
)

total_batches = tf.data.experimental.cardinality(full_dataset).numpy()
train_batches = int(CONFIG["train_split"] * total_batches)
val_batches = int(CONFIG["val_split"] * total_batches)
test_batches = total_batches - train_batches - val_batches

print(f"Total batches: {total_batches}")
print(f"  Training:   {train_batches} batches (~{train_batches * CONFIG['batch_size']} images)")
print(f"  Validation: {val_batches} batches (~{val_batches * CONFIG['batch_size']} images)")
print(f"  Test:       {test_batches} batches (~{test_batches * CONFIG['batch_size']} images)")

train_ds = full_dataset.take(train_batches)
remaining = full_dataset.skip(train_batches)
val_ds = remaining.take(val_batches)
test_ds = remaining.skip(val_batches)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

## 5. Data augmentation

Augmentation is applied only during training, on the fly, on the GPU. Each epoch sees slightly different versions of the same images, which improves generalisation.

### Augmentations chosen

- **Horizontal flip** — maize leaves are roughly symmetric.
- **Rotation (±10%)** — farmers will hold the phone at slightly different angles.
- **Zoom (±10%)** — leaves at different distances from the camera.
- **Contrast (±10%)** and **brightness (±10%)** — field lighting varies.

### What we deliberately do NOT augment

- **Vertical flip** — unnatural orientation.
- **Random colour shift** — hue is diagnostic (orange rust pustules).

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
    layers.RandomBrightness(0.1, value_range=(0, 255)),
], name="data_augmentation")

# Preview a few augmented images
plt.figure(figsize=(12, 12))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        augmented = data_augmentation(tf.expand_dims(images[i % len(images)], 0))
        plt.imshow(tf.cast(augmented[0], tf.uint8).numpy())
        true_class = CONFIG["class_names"][np.argmax(labels[i % len(labels)].numpy())]
        plt.title(f"Augmented — {true_class}", fontsize=10)
        plt.axis("off")
plt.suptitle("Sample augmented training images", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Build the model — MobileNetV2 + classification head

Three layers:

1. **Input pipeline** — augmentation, then ImageNet-style preprocessing.
2. **Backbone** — MobileNetV2 pre-trained on ImageNet, top classification layer removed.
3. **Classification head** — global average pooling → dense → dropout → 4-class softmax.

### Why this architecture

MobileNetV2's depthwise separable convolutions reduce parameters by an order of magnitude versus VGG16 or ResNet50, with negligible loss of accuracy. After quantisation, the final TFLite file is roughly 5 MB.

### Comparing three architectures

We compare three architectures on the same data with identical training schedules. Phase-1 and Phase-2 epoch counts, learning rates, augmentation, and batch size are held constant; only the backbone changes. **EfficientNetB0 is used in place of EfficientNet-Lite0** because the Keras applications module does not ship a native Lite0 variant; their parameter counts and operating characteristics are similar.

In [ ]:
def build_model(architecture, num_classes, input_shape=(224, 224, 3)):
    """Build a five-class classifier with the requested backbone."""

    if architecture == "MobileNetV2":
        base_model = applications.MobileNetV2(
            weights="imagenet", include_top=False, input_shape=input_shape
        )
        preprocess = applications.mobilenet_v2.preprocess_input
    elif architecture == "MobileNetV3Small":
        base_model = applications.MobileNetV3Small(
            weights="imagenet", include_top=False, input_shape=input_shape
        )
        preprocess = applications.mobilenet_v3.preprocess_input
    elif architecture == "EfficientNetLite0":
        # EfficientNet-Lite0 is not shipped natively by Keras applications.
        # Use EfficientNetB0 with the appropriate preprocessing, which is the
        # closest standard equivalent (documented in the markdown above).
        base_model = applications.EfficientNetB0(
            weights="imagenet", include_top=False, input_shape=input_shape
        )
        preprocess = applications.efficientnet.preprocess_input
    else:
        raise ValueError(f"Unknown architecture: {architecture}")

    base_model.trainable = False  # frozen for Phase 1

    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = preprocess(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(CONFIG["dense_units"], activation="relu")(x)
    x = layers.Dropout(CONFIG["dropout_rate"])(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name=f"MaizAI_{architecture}")
    return model, base_model

## 7. Architecture sweep — train all three backbones

The cell below trains each of the three architectures end-to-end (Phase 1 head-only, then Phase 2 fine-tuning) on identical splits and hyperparameters, evaluates each on the held-out test set, and records test accuracy, per-image CPU latency, parameter count and training time.

The two Phase-1 / Phase-2 markdown sections that follow describe what happens inside each iteration of the loop.

> **Runtime:** training all three architectures plus the evaluation, calibration and OOD steps takes roughly **3 hours** on a Colab T4 GPU. Keep the tab active so Colab does not recycle the runtime.

> **Note:** we deliberately do **not** call `tf.keras.backend.clear_session()` between architectures — each trained model is kept in memory so the best one can be converted to TFLite at the end. Three MobileNet-class models fit comfortably in T4 memory; clearing the session would invalidate the earlier stored models.

### Phase 1 — train the classification head

Backbone frozen; only the new head trains. Roughly 1 minute per epoch on T4.

### What to expect

- Training accuracy climbs quickly to 85–95% by epoch 3–4.
- Validation accuracy follows closely, usually within 2–3 points of training accuracy.
- A widening train/val gap would indicate overfitting.

### Phase 2 — fine-tune the top backbone layers

Unfreeze the top ~30% of MobileNetV2 layers and continue at a much lower learning rate.

### Why a 100× smaller learning rate

A high rate during fine-tuning causes "catastrophic forgetting": ImageNet features get overwritten. 1e-5 makes only tiny adjustments per step, preserving prior knowledge.

In [ ]:
ARCHITECTURE_RESULTS = {}

for architecture in CONFIG["architectures"]:
    print(f"\n{'=' * 70}")
    print(f"TRAINING {architecture}")
    print(f"{'=' * 70}\n")

    # NOTE: we do not clear the TF session here — each trained model is kept in
    # ARCHITECTURE_RESULTS so the best one can be exported to TFLite later.
    model, base_model = build_model(architecture, CONFIG["num_classes"])

    # Phase 1 — head only
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=CONFIG["phase1_learning_rate"]),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    phase1_start = time.time()
    phase1_history = model.fit(
        train_ds, validation_data=val_ds,
        epochs=CONFIG["phase1_epochs"], verbose=1,
    )
    phase1_time = time.time() - phase1_start

    # Phase 2 — fine-tune
    base_model.trainable = True
    for layer in base_model.layers[:CONFIG["phase2_unfreeze_from_layer"]]:
        layer.trainable = False

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=CONFIG["phase2_learning_rate"]),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    phase2_start = time.time()
    phase2_history = model.fit(
        train_ds, validation_data=val_ds,
        epochs=CONFIG["phase1_epochs"] + CONFIG["phase2_epochs"],
        initial_epoch=CONFIG["phase1_epochs"], verbose=1,
    )
    phase2_time = time.time() - phase2_start

    # Evaluate on test set
    y_true_arch, y_pred_arch, y_probs_arch = [], [], []
    for images, labels in test_ds:
        preds = model.predict(images, verbose=0)
        y_true_arch.extend(np.argmax(labels.numpy(), axis=1))
        y_pred_arch.extend(np.argmax(preds, axis=1))
        y_probs_arch.append(preds)
    y_true_arch = np.array(y_true_arch)
    y_pred_arch = np.array(y_pred_arch)
    y_probs_arch = np.concatenate(y_probs_arch, axis=0)

    test_accuracy = (y_true_arch == y_pred_arch).mean()

    # Estimate inference latency (single-image)
    sample_image = next(iter(test_ds))[0][:1]
    # Warm-up
    for _ in range(3):
        _ = model.predict(sample_image, verbose=0)
    n_runs = 20
    latency_start = time.time()
    for _ in range(n_runs):
        _ = model.predict(sample_image, verbose=0)
    latency_ms = (time.time() - latency_start) / n_runs * 1000

    # Parameter count
    param_count = model.count_params()

    ARCHITECTURE_RESULTS[architecture] = {
        "model": model,
        "phase1_history": phase1_history.history,
        "phase2_history": phase2_history.history,
        "y_true": y_true_arch,
        "y_pred": y_pred_arch,
        "y_probs": y_probs_arch,
        "test_accuracy": float(test_accuracy),
        "training_time_s": float(phase1_time + phase2_time),
        "inference_latency_ms": float(latency_ms),
        "param_count": int(param_count),
    }

    print(f"\n{architecture} — Test accuracy: {test_accuracy:.4f}  "
          f"Latency: {latency_ms:.1f} ms  Params: {param_count:,}")

print("\nAll architectures trained.")

## 8. Architecture comparison

We compare the three trained backbones on test accuracy, macro F1, parameter count, training time and per-image CPU latency, then select the **best architecture by macro F1**. Everything downstream — training curves, confusion matrix, calibration, OOD test and the exported TFLite model — uses this chosen architecture.

In [ ]:
import pandas as pd

comparison_rows = []
for arch, results in ARCHITECTURE_RESULTS.items():
    # Per-class F1
    f1_report = classification_report(
        results["y_true"], results["y_pred"],
        target_names=CONFIG["class_names"],
        output_dict=True, zero_division=0,
    )

    # Estimate quantised TFLite model size by running the converter
    # but skip the actual save here; we save the chosen model later.
    # Use parameter count as a proxy and document this in the comparison cell.
    comparison_rows.append({
        "Architecture": arch,
        "Test accuracy": round(results["test_accuracy"], 4),
        "Macro F1": round(f1_report["macro avg"]["f1-score"], 4),
        "Params (M)": round(results["param_count"] / 1e6, 2),
        "Training time (min)": round(results["training_time_s"] / 60, 1),
        "Inference (ms, CPU)": round(results["inference_latency_ms"], 1),
    })

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))

# Save for the report
comparison_df.to_csv(f"{CONFIG['output_dir']}/architecture_comparison.csv", index=False)

# Bar chart comparing test accuracy and macro F1
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
archs = comparison_df["Architecture"].tolist()
axes[0].bar(archs, comparison_df["Test accuracy"], color="#3d8b5c")
axes[0].set_title("Test accuracy by architecture")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0.85, 1.0)
for i, v in enumerate(comparison_df["Test accuracy"]):
    axes[0].text(i, v + 0.002, f"{v:.3f}", ha="center", fontsize=10)

axes[1].bar(archs, comparison_df["Inference (ms, CPU)"], color="#a07440")
axes[1].set_title("Inference latency (CPU, per-image)")
axes[1].set_ylabel("Milliseconds")
for i, v in enumerate(comparison_df["Inference (ms, CPU)"]):
    axes[1].text(i, v + 1, f"{v:.0f} ms", ha="center", fontsize=10)

plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/architecture_comparison.png",
            dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: architecture_comparison.csv and architecture_comparison.png")

# Select the best architecture by macro F1 — used by every downstream section.
best_arch = max(
    ARCHITECTURE_RESULTS.keys(),
    key=lambda a: classification_report(
        ARCHITECTURE_RESULTS[a]["y_true"],
        ARCHITECTURE_RESULTS[a]["y_pred"],
        target_names=CONFIG["class_names"],
        output_dict=True, zero_division=0,
    )["macro avg"]["f1-score"],
)
best_results = ARCHITECTURE_RESULTS[best_arch]
# Expose the chosen model's predictions for the evaluation cells below.
y_true = best_results["y_true"]
y_pred = best_results["y_pred"]
y_probs = best_results["y_probs"]
print(f"\nBest architecture by macro F1: {best_arch}")

## 9. Plot training and validation curves

Both phases on a single axis with a vertical line marking the transition.

### What to look for

- A small bump at the Phase 1 → Phase 2 boundary is expected.
- Curves staying close together = good generalisation.
- A widening gap = overfitting.

In [ ]:
# Training/validation curves for the chosen best architecture.
best_p1 = best_results["phase1_history"]
best_p2 = best_results["phase2_history"]
acc = best_p1["accuracy"] + best_p2["accuracy"]
val_acc = best_p1["val_accuracy"] + best_p2["val_accuracy"]
loss = best_p1["loss"] + best_p2["loss"]
val_loss = best_p1["val_loss"] + best_p2["val_loss"]

epochs_range = range(1, len(acc) + 1)
phase_boundary = CONFIG["phase1_epochs"] + 0.5

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(epochs_range, acc, label="Training", linewidth=2)
axes[0].plot(epochs_range, val_acc, label="Validation", linewidth=2)
axes[0].axvline(x=phase_boundary, color="gray", linestyle="--", label="Fine-tuning begins")
axes[0].legend(loc="lower right")
axes[0].set_title(f"Training and Validation Accuracy — {best_arch}", fontsize=13)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, loss, label="Training", linewidth=2)
axes[1].plot(epochs_range, val_loss, label="Validation", linewidth=2)
axes[1].axvline(x=phase_boundary, color="gray", linestyle="--", label="Fine-tuning begins")
axes[1].legend(loc="upper right")
axes[1].set_title(f"Training and Validation Loss — {best_arch}", fontsize=13)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {CONFIG['output_dir']}/training_curves.png ({best_arch})")

## 10. Evaluate on the held-out test set

The test set was set aside before training. Performance on it is our estimate of how the model will perform on new images.

### What gets reported

- **Overall accuracy** — the headline number.
- **Per-class precision** — of the leaves the model called "Common Rust", how many were actually rust?
- **Per-class recall** — of the leaves that actually had rust, how many did the model catch?
- **F1 score** — harmonic mean of precision and recall.
- **Confusion matrix** — which classes get confused with which.

In [ ]:
# Classification report for the chosen best architecture (predictions computed
# during the sweep and exposed as y_true / y_pred / y_probs above). The
# consolidated metrics.json is written by the final summary cell.
y_pred_probs = np.max(y_probs, axis=1)

print("=" * 60)
print(f"CLASSIFICATION REPORT — {best_arch}")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=CONFIG["class_names"], digits=4))

print(f"Test-set size: {len(y_true)} images")
print(f"Mean prediction confidence: {np.mean(y_pred_probs):.4f}")

In [ ]:
cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CONFIG["class_names"],
)

fig, ax = plt.subplots(figsize=(9, 8))
disp.plot(ax=ax, cmap="Blues", xticks_rotation=30, values_format="d")
plt.title(f"Confusion Matrix — {best_arch} (Test Set)", fontsize=13)
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nSaved: {CONFIG['output_dir']}/confusion_matrix.png ({best_arch})")

## 11. Confidence calibration analysis

A well-calibrated classifier reports confidence values that match its actual accuracy: when it says 90% confidence across many predictions, roughly 90% of those should be correct. Modern deep classifiers are known to be poorly calibrated by default — typically overconfident. We measure calibration with a reliability diagram and the expected calibration error (ECE), computed on the test set predictions of the chosen architecture.

In [ ]:
# Select the best architecture by macro F1
best_arch = max(
    ARCHITECTURE_RESULTS.keys(),
    key=lambda a: classification_report(
        ARCHITECTURE_RESULTS[a]["y_true"],
        ARCHITECTURE_RESULTS[a]["y_pred"],
        target_names=CONFIG["class_names"],
        output_dict=True, zero_division=0,
    )["macro avg"]["f1-score"],
)
print(f"Best architecture by macro F1: {best_arch}")

best_results = ARCHITECTURE_RESULTS[best_arch]
y_true_best = best_results["y_true"]
y_pred_best = best_results["y_pred"]
y_probs_best = best_results["y_probs"]
confidences = np.max(y_probs_best, axis=1)
correct = (y_pred_best == y_true_best).astype(int)

# Reliability diagram and ECE
n_bins = CONFIG["calibration_bins"]
bin_boundaries = np.linspace(0, 1, n_bins + 1)
bin_lowers = bin_boundaries[:-1]
bin_uppers = bin_boundaries[1:]

bin_accs = []
bin_confs = []
bin_counts = []
ece = 0.0
for lo, hi in zip(bin_lowers, bin_uppers):
    in_bin = (confidences >= lo) & (confidences < hi)
    count = in_bin.sum()
    if count > 0:
        bin_acc = correct[in_bin].mean()
        bin_conf = confidences[in_bin].mean()
        bin_accs.append(bin_acc)
        bin_confs.append(bin_conf)
        bin_counts.append(count)
        ece += (count / len(confidences)) * abs(bin_acc - bin_conf)
    else:
        bin_accs.append(0.0)
        bin_confs.append((lo + hi) / 2)
        bin_counts.append(0)

print(f"Expected Calibration Error (ECE): {ece:.4f}")

# Plot the reliability diagram
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated", linewidth=1)
ax.bar(bin_lowers, bin_accs, width=1 / n_bins, align="edge",
       edgecolor="#26593b", color="#8ec9a3", alpha=0.85, label="Model")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Predicted confidence")
ax.set_ylabel("Empirical accuracy")
ax.set_title(f"Reliability Diagram — {best_arch}\nECE = {ece:.4f}")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/reliability_diagram.png",
            dpi=150, bbox_inches="tight")
plt.show()

# Save the calibration data for the report
with open(f"{CONFIG['output_dir']}/calibration.json", "w") as f:
    json.dump({
        "architecture": best_arch,
        "ece": float(ece),
        "n_bins": n_bins,
        "bin_lower": [float(x) for x in bin_lowers],
        "bin_accuracy": [float(x) for x in bin_accs],
        "bin_confidence": [float(x) for x in bin_confs],
        "bin_count": [int(x) for x in bin_counts],
    }, f, indent=2)

print(f"Saved: reliability_diagram.png and calibration.json")

## 12. Out-of-distribution test (formal evaluation)

We evaluate how the model handles inputs that are not maize leaves at all. The 100 held-out CIFAR-100 non-plant images (not used during training) are passed through the chosen model. We expect them to be classified as Not_Maize. The rejection rate is the share that are correctly classified as Not_Maize. We also report the mean confidence on these inputs to see whether the model rejects them confidently or with low confidence.

In [ ]:
# Load OOD test images
ood_test_ds = image_dataset_from_directory(
    "./ood_test",
    labels=None,
    image_size=CONFIG["image_size"],
    batch_size=CONFIG["batch_size"],
    shuffle=False,
)

best_model = best_results["model"]

ood_preds = []
ood_confs = []
for images in ood_test_ds:
    probs = best_model.predict(images, verbose=0)
    ood_preds.extend(np.argmax(probs, axis=1))
    ood_confs.extend(np.max(probs, axis=1))

ood_preds = np.array(ood_preds)
ood_confs = np.array(ood_confs)

not_maize_idx = CONFIG["class_names"].index("Not_Maize")
rejected_correctly = (ood_preds == not_maize_idx).sum()
rejection_rate = rejected_correctly / len(ood_preds)

# Per-class breakdown of misclassifications
misclass_breakdown = {}
for i, cls in enumerate(CONFIG["class_names"]):
    misclass_breakdown[cls] = int((ood_preds == i).sum())

print(f"OOD test set: {len(ood_preds)} non-maize images")
print(f"Correctly rejected as Not_Maize: {rejected_correctly} ({rejection_rate:.1%})")
print(f"\nBreakdown of OOD predictions across classes:")
for cls, count in misclass_breakdown.items():
    print(f"  {cls}: {count}")

print(f"\nMean confidence on OOD inputs: {ood_confs.mean():.4f}")
print(f"Mean confidence on correctly-rejected OOD: "
      f"{ood_confs[ood_preds == not_maize_idx].mean():.4f}")

# Save
with open(f"{CONFIG['output_dir']}/ood_evaluation.json", "w") as f:
    json.dump({
        "n_ood_test": len(ood_preds),
        "rejection_rate": float(rejection_rate),
        "mean_confidence": float(ood_confs.mean()),
        "misclassification_breakdown": misclass_breakdown,
    }, f, indent=2)

print(f"\nSaved: ood_evaluation.json")

## 13. Convert the trained model to TensorFlow Lite

Post-training integer quantisation. This is the artifact that ships in the mobile application.

### What quantisation does

| Property | Before | After |
|---|---|---|
| Model size | ~14 MB | ~3.5 MB |
| Inference time on phone | ~150 ms | ~50 ms |
| Accuracy | (baseline) | typically within 0.5–1.0% |

In [ ]:
model_to_export = best_results["model"]
print(f"Converting {best_arch} to TensorFlow Lite...")

# Representative dataset for quantisation calibration (unchanged from v1)
def representative_dataset_gen():
    count = 0
    for images, _ in train_ds:
        for image in images:
            if count >= 100:
                return
            image = tf.cast(tf.expand_dims(image, axis=0), tf.float32)
            yield [image]
            count += 1

converter = tf.lite.TFLiteConverter.from_keras_model(model_to_export)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_gen
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

tflite_model = converter.convert()

TFLITE_PATH = f"{CONFIG['output_dir']}/maize_classifier.tflite"
with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)

size_mb = os.path.getsize(TFLITE_PATH) / (1024 * 1024)
print(f"\nTFLite model ({best_arch}) saved: {TFLITE_PATH}")
print(f"Model size: {size_mb:.2f} MB")

## 14. Sanity-check the TFLite model

Load the `.tflite` back with the TFLite interpreter and verify it produces sensible predictions on test images. If predictions diverge from the Keras model, something went wrong during quantisation.

In [ ]:
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input details:")
print(f"  Name:   {input_details[0]['name']}")
print(f"  Shape:  {input_details[0]['shape']}")
print(f"  Dtype:  {input_details[0]['dtype']}")
print(f"\nOutput details:")
print(f"  Name:   {output_details[0]['name']}")
print(f"  Shape:  {output_details[0]['shape']}")
print(f"  Dtype:  {output_details[0]['dtype']}")

print("\n" + "=" * 60)
print("SAMPLE PREDICTIONS (TFLite model)")
print("=" * 60)
correct = 0
total = 0
for images, labels in test_ds.take(2):
    for i in range(min(6, len(images))):
        img = tf.cast(images[i], tf.uint8).numpy()
        true_idx = np.argmax(labels[i].numpy())
        interpreter.set_tensor(input_details[0]["index"], np.expand_dims(img, axis=0))
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]["index"])
        pred_idx = np.argmax(output)
        marker = "✓" if pred_idx == true_idx else "✗"
        true_name = CONFIG["class_names"][true_idx]
        pred_name = CONFIG["class_names"][pred_idx]
        print(f"  {marker}  True: {true_name:25s}  Predicted: {pred_name}")
        correct += int(pred_idx == true_idx)
        total += 1

print(f"\nSanity check: {correct}/{total} correct")

## 15. Consolidated metrics

Write a single `metrics.json` capturing the chosen architecture, its test metrics, calibration error, OOD rejection rate, model size and the full architecture comparison — the headline numbers for the dissertation's results chapter.

In [ ]:
final_metrics = {
    "chosen_architecture": best_arch,
    "test_accuracy": best_results["test_accuracy"],
    "test_set_size": int(len(best_results["y_true"])),
    "class_names": CONFIG["class_names"],
    "expected_calibration_error": float(ece),
    "ood_rejection_rate": float(rejection_rate),
    "model_size_mb": round(size_mb, 2),
    "inference_latency_ms_cpu": round(best_results["inference_latency_ms"], 1),
    "architecture_comparison": comparison_rows,
    "per_class_metrics": classification_report(
        best_results["y_true"], best_results["y_pred"],
        target_names=CONFIG["class_names"],
        output_dict=True, zero_division=0,
    ),
}

with open(f"{CONFIG['output_dir']}/metrics.json", "w") as f:
    json.dump(final_metrics, f, indent=2)

print(json.dumps(final_metrics, indent=2))

## 16. Download the artifacts

Downloads the full v2 artifact set for the repository:

| File | Purpose |
|---|---|
| `maize_classifier.tflite` | The chosen five-class model, embedded in the mobile app |
| `metrics.json` | Consolidated metrics for the results chapter |
| `architecture_comparison.csv` / `.png` | Three-architecture comparison |
| `reliability_diagram.png` / `calibration.json` | Confidence calibration (ECE) |
| `ood_evaluation.json` | Out-of-distribution rejection results |
| `training_curves.png` | Training curves of the chosen architecture |
| `confusion_matrix.png` | Confusion matrix of the chosen architecture |

Place these in `maizai/ai/models/`, then commit and push.

In [ ]:
from google.colab import files

ARTIFACTS = [
    "maize_classifier.tflite",
    "metrics.json",
    "architecture_comparison.csv",
    "architecture_comparison.png",
    "reliability_diagram.png",
    "calibration.json",
    "ood_evaluation.json",
    "training_curves.png",
    "confusion_matrix.png",
]

print("Downloading artifacts...")
for fname in ARTIFACTS:
    path = f"{CONFIG['output_dir']}/{fname}"
    if os.path.exists(path):
        files.download(path)
    else:
        print(f"  (missing, skipped) {fname}")
print("\nAll artifacts downloaded.")
print("Place them in: maizai/ai/models/")
print("Then commit and push.")